<a href="https://colab.research.google.com/github/Imposon/Trial/blob/main/Unsloth_LLM_FineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning Large Language Models with Unsloth

**What is fine-tuning?**
Fine-tuning is the process of taking a pre-trained language model (like Llama 3.1) and training it further on a specific dataset so it learns to follow instructions, answer questions, or behave in a particular way. Think of it like teaching an already-smart student a new subject.

**Why Unsloth?**
Unsloth is a library that makes fine-tuning 2-5x faster and uses 70% less GPU memory compared to standard methods. This means you can fine-tune large models even on a free Google Colab T4 GPU.

**What we will cover in this notebook:**
1. Environment setup and GPU check
2. Install required libraries
3. Load a 4-bit quantized model (Llama 3.1 8B)
4. Prepare a dataset (from Hugging Face Hub or a local file)
5. Apply LoRA adapters with PEFT (Parameter-Efficient Fine-Tuning)
6. Train using SFTTrainer (Supervised Fine-Tuning)
7. Save and merge model weights
8. Run inference -- compare before vs after fine-tuning
9. (Optional) Push model to Hugging Face Hub

**Requirements:** Google Colab with T4 GPU runtime (free tier works)
**Expected training time:** ~30-45 minutes on a T4 GPU

## 1. Environment Setup and GPU Check

Fine-tuning requires a GPU. Before doing anything else, we need to confirm that our Colab runtime has a GPU attached.

**If you get an error here:**
Go to **Runtime > Change runtime type > Hardware accelerator > T4 GPU**, then click **Save** and re-run this cell.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected! Go to Runtime > Change runtime type > T4 GPU"
    )

gpu_name = torch.cuda.get_device_name(0)
# gpu_memory = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"[OK] GPU detected: {gpu_name}")
# print(f"     Memory: {gpu_memory:.1f} GB")

!nvidia-smi

[OK] GPU detected: Tesla T4
Wed Apr  1 05:46:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------------

## 2. Install Required Libraries

We install the following libraries:
- **unsloth** -- optimized fine-tuning library (the star of this notebook)
- **transformers** -- Hugging Face's core library for working with LLMs
- **trl** -- provides the SFTTrainer for supervised fine-tuning
- **peft** -- implements LoRA (Low-Rank Adaptation) for efficient training
- **accelerate** -- handles distributed training and mixed precision
- **bitsandbytes** -- enables 4-bit quantization to reduce memory usage
- **datasets** -- for loading and processing training data
- **xformers** -- memory-efficient attention implementation

**Note:** This cell may take 2-3 minutes to complete. Watch the output for any errors. If you see version conflicts, restart the runtime (Runtime > Restart runtime) and run from the top.

In [ ]:
# Step 1: Install Unsloth (this will also pull compatible versions of transformers, peft, etc.)
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Step 2: Install remaining dependencies with --no-deps to avoid version conflicts
!pip install --no-deps xformers trl peft accelerate bitsandbytes

# Step 3: Install datasets separately
!pip install datasets

print("\n[OK] All packages installed. If you see any errors above, restart runtime and re-run.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-bscccji_/unsloth_af5f5fba67444f849a91fe80079b898c
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-bscccji_/unsloth_af5f5fba67444f849a91fe80079b898c
  Resolved https://github.com/unslothai/unsloth.git to commit 6984e118eb5b2e885136ce6621ab9ee0eb1eac40
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 29.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 59.7 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 79.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
#free all the gpu memory
torch.cuda.empty_cache()

#delete all teh variables to free up memory
for name in dir():
    if not name.startswith("_"):
        print(globals()[name])
        del globals()[name]



['', 'import torch\n\nif not torch.cuda.is_available():\n    raise RuntimeError(\n        "No GPU detected! Go to Runtime > Change runtime type > T4 GPU"\n    )\n\ngpu_name = torch.cuda.get_device_name(0)\n# gpu_memory = torch.cuda.get_device_properties(0).total_mem / 1e9\nprint(f"[OK] GPU detected: {gpu_name}")\n# print(f"     Memory: {gpu_memory:.1f} GB")\n\nget_ipython().system(\'nvidia-smi\')', '# Step 1: Install Unsloth (this will also pull compatible versions of transformers, peft, etc.)\nget_ipython().system(\'pip install --upgrade pip\')\nget_ipython().system(\'pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"\')\n\n# Step 2: Install remaining dependencies with --no-deps to avoid version conflicts\nget_ipython().system(\'pip install --no-deps xformers trl peft accelerate bitsandbytes\')\n\n# Step 3: Install datasets separately\nget_ipython().system(\'pip install datasets\')\n\nprint("\\n[OK] All packages installed. If you see any errors above, rest

## 3. Import Libraries and Configuration

Now we import everything we need and define configuration variables upfront.

**Key configuration choices explained:**
- `MAX_SEQ_LENGTH = 512` -- how many tokens (roughly words) the model processes at once. Lower = faster training, higher = model sees more context. 512 is a good balance for instruction-tuning.
- `MODEL_NAME` -- we use a pre-quantized 4-bit model from Unsloth's model hub. This drastically reduces memory usage.
- `is_bfloat16_supported` -- Unsloth provides this utility to auto-detect the best floating point precision for your GPU. T4 uses float16, newer GPUs (A100, H100) use bfloat16.

In [ ]:
import os

# Disable TorchDynamo (torch.compile) entirely before importing anything.
# Unsloth's fused cross-entropy loss decorates 'accumulate_chunk' with @torch.compile.
# TorchDynamo then fails during fake-tensor shape inference with:
#   TorchRuntimeError: Expected input batch_size (s97: hint=1024) to match target batch_size (s7: hint=1090)
# This error is NOT suppressed by suppress_errors=True because it originates in
# get_fake_value(), not as a standard graph-break. Disabling Dynamo entirely
# forces all torch.compile-decorated functions to run in eager (normal) mode.
# Training quality is identical; only the JIT compile-time speedup is skipped.
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["UNSLOTH_COMPILE_TORCH"] = "0"   # belt-and-suspenders: also tell unsloth

from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

# -- Configuration --
# You can change these values to experiment with different settings.

MAX_SEQ_LENGTH = 512            # Max tokens per training example (512 is good for speed)
MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-bnb-4bit"  # Pre-quantized 4-bit Llama 3.1
OUTPUT_DIR = "outputs"          # Where training checkpoints are saved
LORA_DIR = "lora_model"        # Where LoRA adapter weights are saved
MERGED_DIR = "merged_model"    # Where the merged full model is saved

print("[OK] All libraries imported successfully")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[OK] All libraries imported successfully


## 4. Load Model with 4-bit Quantization

**What is quantization?**
Normally, model weights are stored as 32-bit or 16-bit floating point numbers. 4-bit quantization compresses these to just 4 bits per weight, reducing memory from ~16 GB down to ~5 GB. The quality loss is minimal thanks to techniques like NF4 (NormalFloat4).

**What are we loading?**
- `model` -- the actual neural network (Llama 3.1 with 8 billion parameters)
- `tokenizer` -- converts text to numbers (tokens) and back

**Parameters:**
- `max_seq_length = 512` -- limits how long each input can be (saves memory)
- `load_in_4bit = True` -- activates 4-bit quantization
- `dtype = None` -- lets Unsloth auto-detect the best precision for your GPU

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,            # Auto-detect (float16 on T4, bfloat16 on A100/H100)
    load_in_4bit=True,     # Use 4-bit quantization to save memory
)

print(f"[OK] Model loaded: {MODEL_NAME}")
print(f"     Vocab size: {model.config.vocab_size}")
print(f"     Max seq length: {MAX_SEQ_LENGTH}")

# Show how much GPU memory the model is using
gpu_mem = torch.cuda.memory_allocated() / 1e9
print(f"     GPU memory used: {gpu_mem:.2f} GB")

==((====))==  Unsloth 2026.3.18: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-bnb-4bit as a legacy tokenizer.


[OK] Model loaded: unsloth/Meta-Llama-3.1-8B-bnb-4bit
     Vocab size: 128256
     Max seq length: 512
     GPU memory used: 5.75 GB


## 5. Dataset Preparation

To fine-tune a model, we need a dataset of examples showing the model what kind of outputs we want. We will use the **Alpaca** format, which has three fields:
- `instruction` -- what the model should do (e.g., "Summarize the text")
- `input` -- optional additional context (e.g., the text to summarize)
- `output` -- the expected response

We show two options below. **Option A** loads a ready-made dataset from Hugging Face. **Option B** shows how to use your own local file.

### Option A: Load from Hugging Face Hub

We use the **Alpaca Cleaned** dataset -- a popular instruction-tuning dataset with ~52K samples. We select only 1,000 samples to keep training fast.

In [ ]:
# Option A: Load from Hugging Face Hub
dataset = load_dataset("yahma/alpaca-cleaned", split="train")

# Shuffle and take 1000 samples for fast training
# You can increase this to 2000-5000 for better results (but slower training)
dataset = dataset.shuffle(seed=42).select(range(10))

print(f"[OK] Dataset loaded: {len(dataset)} samples")
print(f"     Columns: {dataset.column_names}")
print(f"\n-- Sample row --")
print(dataset[3])

[OK] Dataset loaded: 10 samples
     Columns: ['output', 'input', 'instruction']

-- Sample row --
{'output': 'Here are several methods to improve the accuracy of machine learning models:\n\n1. Gathering and cleaning more data: The more data the model is trained on, the more patterns it can recognize, and the better it can generalize to new data. It’s also important to clean the data to handle missing, inconsistent or duplicate information.\n\n2. Data preprocessing: Several preprocessing techniques like normalization, standardization, encoding, and feature selection can improve model accuracy by making the data more suitable for the learning algorithm.\n\n3. Selecting the right algorithm: Different algorithms are suited to different types of data and problems. Experimenting with various algorithms, including ensemble methods that combine several algorithms, can boost accuracy.\n\n4. Hyperparameter tuning: Hyperparameters are the fixed parameters that control the behavior of the learnin

In [ ]:
id = 1

dataset[id]

{'output': 'We solve the equation f(x) = 0 on the domains x ≤ 1 and x > 1.\n\nIf x ≤ 1, then f(x) = -x - 3, so we want to solve -x - 3 = 0. The solution is x = -3, which satisfies x ≤ 1.\n\nIf x > 1, then f(x) = x/2 + 1, so we want to solve x/2 + 1 = 0. The solution is x = -2, but this value does not satisfy x > 1.\n\nTherefore, the only solution is x = -3.',
 'input': '',
 'instruction': 'Let \n f(x) = {[ -x - 3 if x ≤ 1,; x/2 + 1 if x > 1. ].\nFind the sum of all values of x such that f(x) = 0.'}

### Option B: Load from a Local JSON or CSV File (Alternative)

If you have your own dataset, you can upload it to Colab and load it here instead.

**How to upload:** Click the folder icon on the left sidebar in Colab, then click the upload button to upload your file.

Your data should have `instruction`, `input`, and `output` columns. See the expected format in the code below.

**Skip this cell if you used Option A above.**

In [ ]:
# Option B: Load from local files (uncomment the lines below to use)

# -- Expected JSON format --
# Your JSON file should be a list of objects like this:
# [
#   {"instruction": "Summarize the following text.", "input": "Long text here...", "output": "Summary here."},
#   {"instruction": "Translate to French.", "input": "Hello world", "output": "Bonjour le monde"}
# ]

# To load from a JSON file:
# dataset = load_dataset("json", data_files="your_data.json", split="train")

# To load from a CSV file (CSV should have columns: instruction, input, output):
# dataset = load_dataset("csv", data_files="your_data.csv", split="train")

# Select a subset for fast training:
# dataset = dataset.shuffle(seed=42).select(range(min(1000, len(dataset))))



## 6. Format Dataset with Instruction Template

The model does not understand raw JSON. We need to convert each example into a single text string using a **prompt template**. We use the widely-adopted **Alpaca template**:

```
Below is an instruction that describes a task...

### Instruction:
<what to do>

### Input:
<additional context>

### Response:
<expected answer>
```

During training, the model learns to predict the Response given the Instruction and Input. We also append an **EOS (End of Sequence) token** so the model learns when to stop generating.

In [ ]:
# The Alpaca prompt template -- a standard format for instruction-tuning
ALPACA_TEMPLATE = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

# The EOS token tells the model "stop generating here"
EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    """Convert each dataset row into a formatted text string and tokenize it."""
    texts = []
    for instruction, inp, output in zip(
        examples["instruction"], examples["input"], examples["output"]
    ):
        text = ALPACA_TEMPLATE.format(
            instruction=instruction,
            input=inp,
            output=output,
        ) + EOS_TOKEN
        texts.append(text)

    # Explicitly tokenize and strictly truncate to MAX_SEQ_LENGTH
    tokenized = tokenizer(texts, truncation=True, max_length=MAX_SEQ_LENGTH)

    # We keep the raw "text" string so your sanity check in Section 7 still works!
    tokenized["text"] = texts

    return tokenized

# Apply the formatting and tokenization to the entire dataset
dataset = dataset.map(formatting_prompts_func, batched=True)

print("[OK] Dataset formatted and strictly truncated to max_seq_length")

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

[OK] Dataset formatted and strictly truncated to max_seq_length


## 7. Verify Tokenizer Setup

Before training, we verify that the tokenizer is configured correctly. The **tokenizer** converts text into numerical tokens that the model can process.

Key things we check:
- **Pad token** -- used to fill shorter sequences to a uniform length. If not set, we use the EOS token.
- **Token count** -- confirms our samples fit within `max_seq_length = 512`.

Note: The `SFTTrainer` handles the actual tokenization during training. This cell is just a sanity check.

In [ ]:
# Ensure the pad token is set (required for batched training)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Test tokenization on a sample to make sure everything works
sample = dataset[0]["text"]
tokens = tokenizer(sample, truncation=True, max_length=MAX_SEQ_LENGTH)

print(f"[OK] Tokenizer configured")
print(f"     Pad token: '{tokenizer.pad_token}'")
print(f"     Sample token count: {len(tokens['input_ids'])} (max allowed: {MAX_SEQ_LENGTH})")
print(f"     Decoded preview: {tokenizer.decode(tokens['input_ids'][:50])}...")

[OK] Tokenizer configured
     Pad token: '<|finetune_right_pad_id|>'
     Sample token count: 64 (max allowed: 512)
     Decoded preview: <|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Rearrange the following sentence to make the sentence more interesting.

### Input:
She left the...


## 8. Apply LoRA Configuration with PEFT

**What is LoRA?**
LoRA (Low-Rank Adaptation) is a technique that freezes the entire base model and injects small trainable adapter layers into specific parts of the network. Instead of training all 8 billion parameters, we only train ~1-2% of them. This makes training much faster and uses far less memory.

**How does it work?**
Imagine the model's weight matrices as large tables of numbers. LoRA says: "Instead of modifying the whole table, let me add a small correction matrix that captures the new behavior." Mathematically, it decomposes the weight update into two smaller matrices of rank `r`.

**Key parameters:**
| Parameter | Value | What it does |
|-----------|-------|-------------|
| `r` | 16 | Rank of the adapter matrices. Higher = more capacity but more memory |
| `lora_alpha` | 16 | Scaling factor. Usually set equal to `r` |
| `lora_dropout` | 0 | Dropout rate. Unsloth recommends 0 for optimized kernels |
| `target_modules` | q, k, v, o projections | Which layers get LoRA adapters (the attention layers) |
| `use_gradient_checkpointing` | "unsloth" | Trades compute for memory -- enables 2x longer context |

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                    # Rank of LoRA matrices
    lora_alpha=16,           # Scaling factor (usually same as r)
    lora_dropout=0,          # No dropout (Unsloth optimized)
    target_modules=[         # Which layers to add LoRA adapters to
        "q_proj", "k_proj", "v_proj", "o_proj",
    ],
    bias="none",             # Don't train bias terms
    use_gradient_checkpointing="unsloth",  # Saves 30% memory
    random_state=42,         # For reproducibility
)

model.config.use_cache = True
model.config.pretraining_tp = 1

# Show how many parameters we are actually training
trainable, total = model.get_nb_trainable_parameters()
print(f"[OK] LoRA adapters applied")
print(f"     Trainable parameters: {trainable:,} / {total:,}")
print(f"     Trainable %: {100 * trainable / total:.2f}%")
print(f"     (We are only training {100 * trainable / total:.2f}% of the model!)")

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.3.18 patched 32 layers with 32 QKV layers, 32 O layers and 0 MLP layers.


[OK] LoRA adapters applied
     Trainable parameters: 13,631,488 / 8,043,892,736
     Trainable %: 0.17%
     (We are only training 0.17% of the model!)


## 9. Training Setup with SFTTrainer

**What is SFTTrainer?**
SFTTrainer (Supervised Fine-Tuning Trainer) from the TRL library handles the training loop for us. We just need to give it the model, dataset, and hyperparameters.

**Key hyperparameters explained:**
| Parameter | Value | Meaning |
|-----------|-------|---------|
| `per_device_train_batch_size` | 2 | How many examples are processed at once on the GPU |
| `gradient_accumulation_steps` | 4 | Accumulate gradients over 4 mini-batches before updating weights |
| **Effective batch size** | **8** | = 2 x 4. Larger effective batch = more stable training |
| `learning_rate` | 2e-4 | How big each weight update is. Too high = unstable, too low = slow |
| `num_train_epochs` | 1 | How many times we go through the entire dataset |
| `warmup_steps` | 5 | Gradually increase learning rate at the start to avoid instability |
| `fp16` / `bf16` | auto | Mixed precision training -- uses 16-bit floats.for speed. T4 uses fp16, newer GPUs use bf16 |
| `optim` | adamw_8bit | 8-bit optimizer that uses 30% less memory than standard AdamW |

In [ ]:
# Set up training hyperparameters
training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    optim="adamw_8bit",
    seed=42,
)

# Create the trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",        # Column name containing our formatted text
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,               # Use 2 CPU cores for data processing
    packing=False,                    # Don't pack multiple samples into one sequence
    args=training_args,
)

precision = "bf16" if is_bfloat16_supported() else "fp16"
effective_bs = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
print(f"[OK] Trainer configured")
print(f"     Precision: {precision}")
print(f"     Effective batch size: {effective_bs}")
print(f"     Training samples: {len(dataset)}")

[OK] Trainer configured
     Precision: fp16
     Effective batch size: 8
     Training samples: 10


## 10. Execute Training

This is where the actual fine-tuning happens. The trainer will:
1. Loop through the dataset in batches
2. Compute the loss (how wrong the model's predictions are)
3. Backpropagate gradients and update the LoRA adapter weights
4. Log the loss every 10 steps so you can monitor progress

**Expected time:** ~30-45 minutes on a T4 GPU with 1,000 samples.
**What to watch for:** The training loss should decrease over time. If it stays flat or increases, something may be wrong with the data or hyperparameters.

In [ ]:
import time
import torch._dynamo

# Belt-and-suspenders: forcefully disable Dynamo in this session even if
# the env var in the imports cell was set after torch was already loaded.
torch._dynamo.config.disable = True

print("Starting training...")
print("Expected time: ~30-45 minutes on T4 GPU\n")

start_time = time.time()
train_result = trainer.train()
elapsed = time.time() - start_time

print(f"\n[OK] Training complete!")
print(f"     Total time: {elapsed / 60:.1f} minutes")
print(f"     Final loss: {train_result.training_loss:.4f}")
print(f"     Steps completed: {train_result.global_step}")


Starting training...
Expected time: ~30-45 minutes on T4 GPU



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10 | Num Epochs = 1 | Total steps = 2
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 13,631,488 of 8,043,892,736 (0.17% trained)


Step,Training Loss



[OK] Training complete!
     Total time: 0.2 minutes
     Final loss: 1.8544
     Steps completed: 2


### Training Loss Curve

Let us plot how the loss changed during training. A healthy training run shows a **decreasing loss** curve.

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
train_losses = [(log["step"], log["loss"]) for log in log_history if "loss" in log]

if train_losses:
    steps, losses = zip(*train_losses)
    plt.figure(figsize=(8, 4))
    plt.plot(steps, losses, marker="o", markersize=3)
    plt.xlabel("Step")
    plt.ylabel("Training Loss")
    plt.title("Training Loss Curve")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 11. Save LoRA Adapters and Tokenizer

Now we save the trained LoRA adapter weights. These are the **only new weights** we created during training. The files are small (~30-60 MB) because we only trained ~1-2% of the model.

Later, we can load these adapters on top of the original base model to get our fine-tuned model back.

In [ ]:
import os

# Save LoRA adapter weights and tokenizer
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)

# Show what was saved
print(f"[OK] LoRA adapters saved to '{LORA_DIR}/'")
print(f"     Saved files:")
for f in sorted(os.listdir(LORA_DIR)):
    size = os.path.getsize(os.path.join(LORA_DIR, f)) / 1e6
    print(f"     - {f} ({size:.1f} MB)")

[OK] LoRA adapters saved to 'lora_model/'
     Saved files:
     - README.md (0.0 MB)
     - adapter_config.json (0.0 MB)
     - adapter_model.safetensors (54.6 MB)
     - tokenizer.json (17.2 MB)
     - tokenizer_config.json (0.0 MB)


## 12. Merge LoRA Weights into Base Model (Optional)

You have two choices for how to save and deploy your model:

| Approach | File Size | Pros | Cons |
|----------|-----------|------|------|
| **LoRA adapters** (from step 11) | ~30-60 MB | Small, easy to share, can swap adapters | Need base model + adapter at inference time |
| **Merged model** (this step) | ~16 GB | Self-contained, simpler to deploy | Large file, harder to share |

The cell below merges the LoRA weights back into the base model and saves a complete standalone model in float16 format. **This step is optional** -- you can skip it if you only want the adapters.

In [ ]:
# # Merge LoRA weights into the base model and save as float16
# # This creates a standalone model that does not need the adapter files
# model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")

# print(f"[OK] Merged model saved to '{MERGED_DIR}/'")
# for f in sorted(os.listdir(MERGED_DIR)):
#     size = os.path.getsize(os.path.join(MERGED_DIR, f)) / 1e6
#     print(f"     - {f} ({size:.1f} MB)")

## 13. Inference -- Set Up the Fine-Tuned Model for Generation

Now the fun part -- let us use our fine-tuned model to generate responses!

We call `FastLanguageModel.for_inference(model)` to switch the model from training mode to inference mode. Unsloth optimizes this to run **2x faster** than standard inference.

We also define a helper function `generate_response()` that:
1. Formats the prompt using our Alpaca template
2. Tokenizes it and sends it to the GPU
3. Generates a response using sampling (temperature + top_p)
4. Decodes the output back to text

In [ ]:
import gc
import torch
from unsloth import FastLanguageModel

# ── Step 1: Free GPU memory from training ─────────────────────────────────────
# Delete the trainer and the training copy of the model so their GPU memory
# is returned to the pool before we load the inference copy.
for _var in ("trainer", "model", "tokenizer"):
    if _var in globals():
        del globals()[_var]

gc.collect()
torch.cuda.empty_cache()
print(f"[OK] GPU memory freed: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated now")

# ── Step 2: Reload base model + LoRA adapter from saved weights ───────────────
# FastLanguageModel.from_pretrained detects adapter_config.json in MERGED_DIR and
# automatically loads the base model + merges the LoRA adapter on top.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MERGED_DIR,          # Load from the adapter directory we saved earlier
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
model.eval()  # Switch to eval mode (disables dropout, etc.)

print(f"[OK] Model reloaded from '{MERGED_DIR}'")
print(f"     GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# ── Step 3: Define inference helper ───────────────────────────────────────────
def generate_response(instruction, input_text="", max_new_tokens=256):
    """Generate a response from the reloaded fine-tuned model."""
    prompt = ALPACA_TEMPLATE.format(
        instruction=instruction,
        input=input_text,
        output="",
    )
    prompt = prompt.rsplit("### Response:\n", 1)[0] + "### Response:\n"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (skip the prompt tokens)
    input_length = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][input_length:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print("[OK] Inference function ready -- model loaded from saved LoRA adapters")


[OK] GPU memory freed: 1.10 GB allocated now
==((====))==  Unsloth 2026.3.18: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The tokenizer you are loading from 'merged_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from 'merged_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Unsloth: Will load merged_model as a legacy tokenizer.


[OK] Model reloaded from 'merged_model'
     GPU memory used: 6.84 GB
[OK] Inference function ready -- model loaded from saved LoRA adapters


## 14. Inference -- Test the Fine-Tuned Model

Let us test the model with several different types of prompts. Since we fine-tuned on instruction-following data, the model should now produce structured, helpful responses that follow the given instructions.

We test with three types of tasks:
1. **Explanation** -- can the model explain a concept clearly?
2. **Summarization** -- can it condense information?
3. **Code generation** -- can it write code from a description?

In [ ]:
ALPACA_TEMPLATE = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""


# Define test prompts covering different task types
test_prompts = [
    {
        "instruction": "Explain the concept of machine learning in simple terms.",
        "input": "",
    },
    {
        "instruction": "Summarize the following text in one sentence.",
        "input": "Artificial intelligence has transformed many industries including healthcare, "
                 "finance, transportation, and education. AI systems can now diagnose diseases, "
                 "predict market trends, drive autonomous vehicles, and personalize learning experiences.",
    },
]

# Run inference on each test prompt
for i, prompt in enumerate(test_prompts, 1):
    print("=" * 60)
    print(f"Test {i}")
    print("=" * 60)
    print(f"Instruction: {prompt['instruction']}")
    if prompt["input"]:
        print(f"Input: {prompt['input'][:100]}...")
    print(f"\n[Model Response]:")
    response = generate_response(prompt["instruction"], prompt["input"])
    print(response)
    print()

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test 1
Instruction: Explain the concept of machine learning in simple terms.

[Model Response]:


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Machine Learning is a subfield of artificial intelligence (AI) that focuses on developing algorithms and models that can learn from data and improve their performance over time without being explicitly programmed.

It involves using statistical techniques to identify patterns and make predictions based on large datasets. Machine learning models are trained using labeled examples and then used to make decisions or predictions for new, unlabeled data.

Machine learning has revolutionized many industries by enabling automated decision-making processes, such as self-driving cars, fraud detection, personalized marketing campaigns, etc. It's also used in scientific research to analyze complex biological data like DNA sequencing results or medical images.

Test 2
Instruction: Summarize the following text in one sentence.
Input: Artificial intelligence has transformed many industries including healthcare, finance, transportatio...

[Model Response]:
AI systems have transformed numerous sectors

---

## Done!

You have successfully fine-tuned **Llama 3.1 8B** using **Unsloth** with **LoRA/PEFT**.

### What you learned:
- How to load a quantized model to save GPU memory
- How to prepare and format a dataset for instruction-tuning
- How LoRA works and why it makes fine-tuning efficient
- How to train with SFTTrainer and monitor the loss
- How to save, merge, and use the fine-tuned model

### What to try next:
- Increase training data to 2,000-10,000 samples for better results
- Train for more epochs (`num_train_epochs = 3`)
- Try a different base model (Mistral 7B, Gemma, Phi-3)
- Use your own custom dataset for a specific use case
- Deploy the merged model with vLLM or Ollama
- Push to Hugging Face Hub for sharing

### Resources:
- [Unsloth Reference Notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_%288B%29-Alpaca.ipynb#scrollTo=MKX_XKs_BNZR)
- [Unsloth GitHub](https://github.com/unslothai/unsloth)
- [Hugging Face TRL docs](https://huggingface.co/docs/trl)
- [LoRA paper (Hu et al., 2021)](https://arxiv.org/abs/2106.09685)
- [QLoRA paper (Dettmers et al., 2023)](https://arxiv.org/abs/2305.14314)
